# E-commerce Customer Retention Analysis

## Part 2 — Machine Learning Models

### Objective

Develop, compare and evaluate supervised Machine Learning models capable of predicting customer recurrence using behavioral and transactional data from the Olist marketplace.

### Dataset

The analytical dataset generated during Part 1 of the project.

### Technologies

- Python
- Pandas
- Scikit-learn
- NumPy
- Matplotlib

### Technologies

- Python
- Pandas
- NumPy
- Scikit-learn
- Matplotlib
- Seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
import sys
!{sys.executable} -m pip install -U pandas-profiling[notebook]

In [ ]:
import ydata_profiling as pp
from ydata_profiling import ProfileReport

In [ ]:
from datetime import datetime
from datetime import timedelta

In [ ]:
url1= 'https://raw.githubusercontent.com/Jorgelina-Etchevest/Olist/main/olist_customers_dataset.csv'
df1=pd.read_csv(url1)
url2='https://raw.githubusercontent.com/Jorgelina-Etchevest/Olist/main/olist_order_items_dataset.csv'
df2=pd.read_csv(url2)
url3 = 'https://raw.githubusercontent.com/Jorgelina-Etchevest/Olist/main/olist_order_payments_dataset.csv'
df3=pd.read_csv(url3)
url4='https://raw.githubusercontent.com/Jorgelina-Etchevest/Olist/main/olist_order_reviews_dataset.csv'
df4=pd.read_csv(url4)
url5='https://raw.githubusercontent.com/Jorgelina-Etchevest/Olist/main/olist_orders_dataset.csv'
df5=pd.read_csv(url5)
url6='https://raw.githubusercontent.com/Jorgelina-Etchevest/Olist/main/olist_products_dataset.csv'
df6=pd.read_csv(url6)
url7='https://raw.githubusercontent.com/Jorgelina-Etchevest/Olist/main/olist_sellers_dataset.csv'
df7=pd.read_csv(url7)
url8='https://raw.githubusercontent.com/Jorgelina-Etchevest/Olist/main/product_category_name_translation.csv'
df8=pd.read_csv(url8)




In [ ]:
df_merged = pd.merge(df2, df3, how='outer', on='order_id')
df_merged = pd.merge(df_merged, df4, how='outer', on='order_id')
df_merged = pd.merge(df_merged, df5, how='outer',on='order_id')
df_merged = pd.merge(df_merged, df1, how='outer',on='customer_id')
df_merged = pd.merge(df_merged, df6, how='outer',on='product_id')
df_merged = pd.merge(df_merged, df7, how='outer',on='seller_id')
df_merged = pd.merge(df_merged, df8, how='outer',on='product_category_name')

df_merged.sample(6)

In [ ]:
df_merged.shape

In [ ]:
df_merged.columns

In [ ]:
pd.options.display.float_format = '{:.2f}'.format #coloca 2 decimales
df_merged.describe()

In [ ]:
df_merged.describe(include='object')

In [ ]:
df_merged.duplicated().sum()

In [ ]:
df_merged.isnull().sum()

In [ ]:
def status(data):

    data2=data

    # total de filas
    tot_rows=len(data2)

    # total de nan
    d2=data2.isnull().sum().reset_index()
    d2.columns=['variable', 'q_nan']

    # porcentaje de nan
    d2[['p_nan']]=d2[['q_nan']]/tot_rows

    # numero de ceros
    d2['q_zeros']=(data2==0).sum().values

    # porcentaje de zeros
    d2['p_zeros']=d2[['q_zeros']]/tot_rows

    # total de valores unicos
    d2['unique']=data2.nunique().values

    # obtiene tipos de datos por columna
    d2['type']=[str(x) for x in data2.dtypes.values]

    return(d2)

In [ ]:
pd.options.display.float_format = '{:,.4f}'.format # 1 decimal
status(df_merged)

In [ ]:
#Ya que usaremos las categorías en inglés, primero completaremos aquellas en las que
#sí tenemos datos en portugués. Luego eliminaremos la variable en portugués.
df_cat = df_merged[['product_category_name_english', 'product_category_name']]
df_cat.head()
df_cat.info()

In [ ]:
df_cat.isnull().sum()
#Existen 25 entradas en las que no se identifica la traducción

In [ ]:
df_cat[df_cat['product_category_name_english'].isnull()]
#Observando el df formado por estas dos variables, podemos encontrar dos categorías
#no traducidas

In [ ]:
df_cat[df_cat['product_category_name'] == 'portateis_cozinha_e_preparadores_de_alimentos'].count()
#15/25 pertenecen a esta categoría

In [ ]:
df_cat[df_cat['product_category_name'] == 'pc_gamer'].count()
#Las 10 restantes a esta otra categoría

In [ ]:
df_pcgamer = df_merged['product_category_name_english'].index.isin(range(118713,118723))
df_merged.loc[df_pcgamer & df_merged['product_category_name_english'].isnull(), 'product_category_name_english'] = 'pc_gamer'
df_pd = df_merged['product_category_name_english'].index.isin(range(119128,119143))
df_merged.loc[df_pd & df_merged['product_category_name_english'].isnull(), 'product_category_name_english'] = 'portable_kitchen_and_food_preparation_devices'
df_merged[df_merged['product_category_name_english'].isnull()]
#Podemos ver que actualmente la cantidad de perdidos equipara a las categorias en portugues

In [ ]:
df_estados=pd.read_csv('https://raw.githubusercontent.com/Jorgelina-Etchevest/Olist/main/estados.csv', sep=';')
df_estados

In [ ]:
df_estados.index = df_estados.ABREVIACION
di_df_estados = df_estados['ESTADO'].to_dict()
di_df_estados

In [ ]:
df_merged['estado_cl'] = df_merged['customer_state'].replace(di_df_estados)
df_merged['estado_v'] = df_merged['customer_state'].replace(di_df_estados)
df_merged.drop(['customer_state','seller_state'], axis=1, inplace=True)

In [ ]:
#Procederemos a ajustar el dtype de las variables de naturaleza datetime
df_merged['shipping_limit_date'] = pd.to_datetime(df_merged['order_purchase_timestamp'])
df_merged['order_purchase_timestamp'] = pd.to_datetime(df_merged['order_purchase_timestamp'])
df_merged['order_approved_at'] = pd.to_datetime(df_merged['order_approved_at'])
df_merged['order_delivered_carrier_date'] = pd.to_datetime(df_merged['order_delivered_carrier_date'])
df_merged['order_delivered_customer_date'] = pd.to_datetime(df_merged['order_delivered_customer_date'])
df_merged['order_estimated_delivery_date'] = pd.to_datetime(df_merged['order_estimated_delivery_date'])

In [ ]:
df_merged.dtypes

In [ ]:
df_merged['volumen'] = df_merged['product_length_cm'] * df_merged['product_height_cm'] * df_merged['product_width_cm']
df_merged.drop(['product_length_cm','product_height_cm','product_width_cm'],axis=1,inplace=True)

In [ ]:
import sys
!{sys.executable} -m pip install -U pandas-profiling[notebook]

In [ ]:
#Importar libreria
import ydata_profiling as pp  #importamos el módulo y asigno alias
from ydata_profiling import ProfileReport

In [ ]:
#Creamos un nuevo dataframe para tener a mano datos del df original de ser necesarios en df_merged
df_merged1 = df_merged.drop(['review_comment_title','review_comment_message','review_id','review_creation_date','review_answer_timestamp','product_category_name'], axis=1)
df_merged1['product_category'] = df_merged1['product_category_name_english']
df_merged1.drop(['product_category_name_english'], axis=1, inplace=True)
df_merged1.shape

In [ ]:
df_merged1.info()

In [ ]:
df_merged1.isnull().sum()

In [ ]:
df_merged1[df_merged1['product_id'].isnull()]

In [ ]:
df_merged1.drop(index=range(118310, 119143), inplace=True)

In [ ]:
df_merged1.isnull().sum()

In [ ]:
#Ahora queremos saber si los orders_id tienen alguna relación con order_purchase_timestamp
df_ord = df_merged1[['order_id', 'order_purchase_timestamp']]
df_ord = df_ord.sort_values('order_purchase_timestamp').reset_index(drop=True)
df_ord.head()

In [ ]:
are_ids_ordered = df_ord['order_id'].is_monotonic_increasing
print("¿Los IDs de las órdenes están en el mismo orden que las fechas?:", are_ids_ordered)
#De esta manera comprobamos que no siguen un orden

In [ ]:
mediana_approved = df_merged1['order_approved_at'].median()
mediana_carrier = df_merged1['order_delivered_carrier_date'].median()
mediana_customer = df_merged1['order_delivered_customer_date'].median()
df_merged1['order_approved_at'] = df_merged1['order_approved_at'].fillna(mediana_approved)
df_merged1['order_delivered_carrier_date'] = df_merged1['order_delivered_carrier_date'].fillna(mediana_carrier)
df_merged1['order_delivered_customer_date'] = df_merged1['order_delivered_customer_date'].fillna(mediana_customer)
df_merged1.info()

In [ ]:
mediana_sequential = df_merged1['payment_sequential'].median()
mediana_installments = df_merged1['payment_installments'].median()
mediana_value = df_merged1['payment_value'].median()
mediana_score = df_merged1['review_score'].median()
mediana_name_lenght = df_merged1['product_name_lenght'].median()
mediana_desc_lenght = df_merged1['product_description_lenght'].median()
mediana_ph = df_merged1['product_photos_qty'].median()
mediana_weight = df_merged1['product_weight_g'].median()
mediana_volumen = df_merged1['volumen'].median()

df_merged1['payment_sequential'] = df_merged1['payment_sequential'].fillna(mediana_sequential)
df_merged1['payment_installments'] = df_merged1['payment_installments'].fillna(mediana_installments)
df_merged1['payment_value'] = df_merged1['payment_value'].fillna(mediana_value)
df_merged1['review_score'] = df_merged1['review_score'].fillna(mediana_score)
df_merged1['product_name_lenght'] = df_merged1['product_name_lenght'].fillna(mediana_name_lenght)
df_merged1['product_description_lenght'] = df_merged1['product_description_lenght'].fillna(mediana_desc_lenght)
df_merged1['product_photos_qty'] = df_merged1['product_photos_qty'].fillna(mediana_ph)
df_merged1['product_weight_g'] = df_merged1['product_weight_g'].fillna(mediana_weight)
df_merged1['volumen'] = df_merged1['volumen'].fillna(mediana_volumen)

df_merged1.info()

In [ ]:
df_merged1['payment_type'].value_counts()

In [ ]:
df_merged1['payment_type'] = df_merged1['payment_type'].fillna('credit_card')
df_merged1['product_category'] = df_merged1['product_category'].fillna('unknown')

df_merged1.isnull().sum()

In [ ]:
df_merged1.drop_duplicates(inplace=True)

In [ ]:
#En este punto, por si es necesario reflotar el df_merged1, renombraremos el df
df_merged2 = df_merged1

In [ ]:
#rango intercuartilicO (50% central)
RI_p_v = np.percentile(df_merged2 ['payment_value'], 75) - np.percentile(df_merged2['payment_value'], 25)
RI_p_v

In [ ]:
#Límites inferior y superior
LI_p_v = np.percentile(df_merged2 ['payment_value'], 25) - 3 * RI_p_v
print('Limite inferior =',LI_p_v)
LS_p_v =np.percentile(df_merged2 ['payment_value'], 75) + 3 * RI_p_v
LS_p_v
print('Limite superior =',LS_p_v)

In [ ]:
#Si quisieramos mantener las observaciones por debajo del limite superior
#deberíamos eliminar 4980 columnas
outliers_p_v = df_merged2[df_merged2['payment_value'] >574.72]
porcentaje_outliers_p_v = (len(outliers_p_v) / len(df_merged2)) * 100
print(outliers_p_v.shape)
print('La proporción de valores outliers es=',porcentaje_outliers_p_v,'%')

In [ ]:
#rango intercuartilicO (50% central)
RI_f_v = np.percentile(df_merged2 ['freight_value'], 75) - np.percentile(df_merged2['freight_value'], 25)
RI_f_v

In [ ]:
#Límites inferior y superior
LI_f_v = np.percentile(df_merged2 ['freight_value'], 25) - 3 * RI_f_v
print('Limite inferior =',LI_f_v)
LS_f_v =np.percentile(df_merged2 ['freight_value'], 75) + 3 * RI_f_v
print('Limite superior =',LS_f_v)

In [ ]:
#Si quisieramos mantener las observaciones por debajo del limite superior
#deberíamos eliminar 5830 columnas
outliers_f_v = df_merged2[df_merged2['freight_value'] >45.52]
porcentaje_outliers_f_v = (len(outliers_f_v) / len(df_merged2)) * 100
print(outliers_f_v.shape)
print('La proporción de valores outliers es=',porcentaje_outliers_f_v,'%')

In [ ]:
#rango intercuartilicO (50% central)
RI_v = np.percentile(df_merged2 ['volumen'], 75) - np.percentile(df_merged2['volumen'], 25)
RI_v

In [ ]:
#Límites inferior y superior
LI_v = np.percentile(df_merged2 ['volumen'], 25) - 3 * RI_v
print('Limite inferior =',LI_v)
LS_v =np.percentile(df_merged2 ['volumen'], 75) + 3 * RI_v
print('Limite superior =',LS_v)

In [ ]:
#Si quisieramos mantener las observaciones por debajo del limite superior
#deberíamos eliminar 4566 columnas
outliers_v = df_merged2[df_merged2['volumen'] >65688]
porcentaje_outliers_v = (len(outliers_v) / len(df_merged2)) * 100
print(outliers_v.shape)
print('La proporción de valores outliers es=',porcentaje_outliers_v,'%')

In [ ]:
df_merged2.info()

After cleaning the dataset, handling missing values and removing outliers, new analytical features were created to improve the predictive performance of the Machine Learning models.



# Economic Analysis

In [ ]:
#Dentro de 117891 observaciones, Olist cuenta con un total de 98666 órdenes únicas

print("El total de ventas es=", df_merged2['order_id'].nunique())

In [ ]:
#Dentro de este número de órdenes, los clientes únicos son 95420
print("El total de cliente únicos es=", df_merged2['customer_unique_id'].nunique())

## Customer Return Rate

The customer return rate was calculated to estimate the proportion of recurring customers within the dataset. This metric provides a baseline for evaluating customer retention and supports the development of predictive models.

In [ ]:
#Para calcular la tasa de retorno, primero eliminamos los duplicados de order_id
# Esto permitirá que no cuente dos veces la misma orden
frec = df_merged2[['customer_unique_id','order_id']].drop_duplicates()

In [ ]:
#Luego, agrupamos los id de clientes y de ordenes y creamos la variable frecuencia
#Frecuencia nos dirá cuantas veces compró el mismo cliente
frec = frec.groupby('customer_unique_id')['order_id'].count()
frec = frec.reset_index()
frec.columns = ['customer_unique_id', 'frecuencia']
frec.head()

In [ ]:
#Posteriormente, creamos una máscara con los clientes que compraron más de una vez
# y sacamos la cantidad de clientes que volvieron a comprar/fieles
frec_2 = frec[frec['frecuencia'] >= 2]
print("La cantidad de clientes que volvieron a comprar es =",frec_2.shape[0])

In [ ]:
#Tasa de retorno / proporción de cliente "VUELVE"
tasa_retorno = frec_2.shape[0] / frec.shape[0]
print(f"La tasa de retorno es = {tasa_retorno*100:.2F}%")

In [ ]:
#En contraposición se puede cálcular la tasa de clientes "NO VUELVE"
print(f"La tasa de clientes NO VUELVE es = {(1-tasa_retorno)*100:.2F}%")

## Average Shipping Cost

Average shipping cost was analyzed because logistics expenses are expected to influence customer satisfaction and repurchase behavior.

In [ ]:
#Valor de flete de promedio
print("El valor de flete promedio es =",df_merged2['freight_value'].mean())

In [ ]:
df_flete = df_merged2

In [ ]:
df_flete['flete_promedio'] = df_flete.groupby('order_id')['freight_value'].transform('mean')
df_flete.head()

In [ ]:
frec2 = df_flete.groupby('customer_unique_id')['order_id'].nunique().reset_index()
frec2.columns = ['customer_unique_id', 'frec2']

In [ ]:
frec2['tipo_cliente'] = frec2['frec2'].apply(lambda x: 'VUELVE' if x > 1 else 'NO VUELVE')

In [ ]:
df_flete = df_flete.merge(frec2[['customer_unique_id', 'tipo_cliente']], on='customer_unique_id', how='left')

In [ ]:
flete_prom_clase = df_flete.groupby('tipo_cliente')['freight_value'].mean().reset_index()
flete_prom_clase.columns = ['tipo_cliente', 'flete_promedio']
print(flete_prom_clase)
#Esto arroja un primer indicio de que los clientes recurrentes pagan un flete menor

In [ ]:
# Obtener los promedios de flete para cada tipo de cliente
flete_vuelve = flete_prom_clase.loc[flete_prom_clase['tipo_cliente'] == 'VUELVE', 'flete_promedio'].values[0]
flete_no_vuelve = flete_prom_clase.loc[flete_prom_clase['tipo_cliente'] == 'NO VUELVE', 'flete_promedio'].values[0]

porcentaje_menos = ((flete_no_vuelve - flete_vuelve) / flete_no_vuelve) * 100

print(f"El cliente vuelve paga un {porcentaje_menos:.2f}% menos que el cliente no vuelve.")

The initial business analysis provided key insights into customer purchasing behavior, shipping costs and order characteristics, serving as the foundation for feature engineering and predictive modeling.

# Feature Engineering

Feature engineering was performed to create new analytical variables capable of improving the predictive performance of the Machine Learning models.

## New Feature: Delivery Delay (Days)

Delivery delay was calculated as the difference between the estimated and actual delivery dates, providing an indicator of customer experience.

In [ ]:
df_merged2['dias_entre_compra_y_entrega_estimada'] = df_merged2['order_estimated_delivery_date'] -df_merged2['order_purchase_timestamp']

In [ ]:
df_merged2['dias_entre_compra_y_entrega_estimada'] = df_merged2['dias_entre_compra_y_entrega_estimada'].dt.days

In [ ]:
df_merged2.head()

## Payment Method Encoding (Dummy Variables)

Categorical payment methods were transformed into dummy variables to enable their use in Machine Learning algorithms.

In [ ]:
medio_pago_Dummy = pd.get_dummies(df_merged2["payment_type"],dtype=int)

In [ ]:
medio_pago_Dummy.info()

In [ ]:
NuevaDf = pd.concat([
    df_merged2.drop("payment_type", axis = 1),
    medio_pago_Dummy
], axis = 1)

In [ ]:
NuevaDf.info()

## Average Sales by State

Customer states were replaced by their corresponding average sales values to incorporate regional purchasing behavior into the predictive models.

In [ ]:
NuevaDf['estado_cl'].unique()

In [ ]:
venta_prom_estado_cliente = NuevaDf.groupby('estado_cl')['payment_value'].mean()
venta_prom_estado_cliente = venta_prom_estado_cliente.reset_index()
venta_prom_estado_cliente.columns = ['estado_cl', 'Venta_promedio_Estado_Cliente']
print(venta_prom_estado_cliente.shape)
venta_prom_estado_cliente.head(10)

In [ ]:
union = pd.merge(NuevaDf, venta_prom_estado_cliente,  how='inner',on='estado_cl')
union.head(20)

In [ ]:
union.info()

# Dataset Preparation for Machine Learning

## Hypothesis

**Hypothesis:** Olist has a large number of customers who made only **one purchase** and did not return during the period under analysis. Implementing targeted **customer retention strategies** and personalized marketing campaigns could encourage these customers to make repeat purchases, increasing customer lifetime value and overall revenue.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.weightstats import ttest_ind

## Total Amount Paid

In [ ]:
compras = union.groupby('customer_unique_id')['payment_value'].sum()
compras = compras.reset_index()
compras.columns = ['customer_unique_id', 'monto_pagado']
print(compras.shape)
compras.head()

## Purchase Frequency

In [ ]:
frecuencia = union[['customer_unique_id', 'order_id']].drop_duplicates()

In [ ]:
frecuencia = frecuencia.groupby('customer_unique_id')['order_id'].count()
frecuencia = frecuencia.reset_index()
frecuencia.columns = ['customer_unique_id', 'frecuencia_compra']
print(frecuencia.shape)
frecuencia.head()

## Purchase Frequency Analysis

In [ ]:
frecuencia['frecuencia_compra'].value_counts()

In [ ]:
frecuencia['frecuencia_compra'].unique()

In [ ]:
frecuencia[frecuencia['customer_unique_id'] == '9a736b248f67d166d2fbb006bcb877c3']

In [ ]:
frecuencia['frecuencia_compra'].unique()

In [ ]:
df_merged2['order_id'].value_counts()

In [ ]:
df_merged2[df_merged2['order_id'] == '895ab968e7bb0d5659d16cd74cd1650c']

In [ ]:
unionData = pd.merge(compras, frecuencia,  how='inner',on='customer_unique_id')
unionData.head()

## Average Sales by Customer State

In [ ]:
union1 = union.groupby('customer_unique_id')['Venta_promedio_Estado_Cliente'].mean()
union1 = union1.reset_index()
union1.columns = ['customer_unique_id', 'venta_promedio_estado_cliente']
print(union1.shape)
union1.head(20)

In [ ]:
union2 = pd.merge(unionData, union1,  how='inner',on='customer_unique_id')
union2.head(20)

## Payment Method Dummy Variables

In [ ]:
tipo_pago_boleto = NuevaDf.groupby('customer_unique_id')['boleto'].min()
tipo_pago_boleto = tipo_pago_boleto.reset_index()
tipo_pago_boleto.columns = ['customer_unique_id', 'boleto']
print(tipo_pago_boleto.shape)
tipo_pago_boleto.head()

In [ ]:
union3 = pd.merge(union2, tipo_pago_boleto,  how='inner',on='customer_unique_id')
union3.head(20)


In [ ]:
tipo_pago_credito = NuevaDf.groupby('customer_unique_id')['credit_card'].min()
tipo_pago_credito = tipo_pago_credito.reset_index()
tipo_pago_credito.columns = ['customer_unique_id', 'tarjeta_crédito']
print(tipo_pago_credito.shape)
tipo_pago_credito.head(25)

In [ ]:
union4 = pd.merge(union3, tipo_pago_credito,  how='inner',on='customer_unique_id')
union4.head(20)

In [ ]:
tipo_pago_debito = NuevaDf.groupby('customer_unique_id')['debit_card'].min()
tipo_pago_debito = tipo_pago_debito.reset_index()
tipo_pago_debito.columns = ['customer_unique_id', 'tarjeta_débito']
print(tipo_pago_debito.shape)
tipo_pago_debito.head(25)

In [ ]:
union5 = pd.merge(union4, tipo_pago_debito,  how='inner',on='customer_unique_id')
union5.head(20)

In [ ]:
tipo_voucher = NuevaDf.groupby('customer_unique_id')['voucher'].min()
tipo_voucher = tipo_voucher.reset_index()
tipo_voucher.columns = ['customer_unique_id', 'voucher']
print(tipo_voucher.shape)
tipo_voucher.head(25)

In [ ]:
union6 = pd.merge(union5, tipo_voucher,  how='inner',on='customer_unique_id')
union6.head(20)

## Average Price

In [ ]:
precio_medio = NuevaDf.groupby('customer_unique_id')['price'].mean()

In [ ]:
precio_medio = precio_medio.reset_index()
precio_medio.columns = ['customer_unique_id', 'precio_promedio']
print(precio_medio.shape)
precio_medio.head()

In [ ]:
union7 = pd.merge(union6, precio_medio,  how='inner',on='customer_unique_id')
union7.head(20)

## Average Volume

In [ ]:
volumen_medio = NuevaDf.groupby('customer_unique_id')['volumen'].mean()
volumen_medio = volumen_medio.reset_index()
volumen_medio.columns = ['customer_unique_id', 'volumen_promedio']
print(volumen_medio.shape)
volumen_medio.head()

In [ ]:
union8 = pd.merge(union7, volumen_medio,  how='inner',on='customer_unique_id')
union8.head(5)

## Average Delivery Delay

In [ ]:
demora = NuevaDf.groupby('customer_unique_id')['dias_entre_compra_y_entrega_estimada'].mean()
demora = demora.reset_index()
demora.columns = ['customer_unique_id', 'demora_promedio_días']
print(demora.shape)
demora.head()

In [ ]:
union9 = pd.merge(union8, demora,  how='inner',on='customer_unique_id')
union9.head(5)

## Total Shipping Cost

In [ ]:
valor_flete = NuevaDf.groupby('customer_unique_id')['freight_value'].sum()
valor_flete = valor_flete.reset_index()
valor_flete.columns = ['customer_unique_id', 'valor_total_flete']
print(valor_flete.shape)
valor_flete.head()

In [ ]:
union10 = pd.merge(union9, valor_flete,  how='inner',on='customer_unique_id')
union10.head(5)

In [ ]:
print(union10.columns)

## Target Variable: Binary Purchase Frequency

Purchase frequency was transformed into a binary target variable representing returning and non-returning customers.


In [ ]:
union10['frecuencia_binaria'] = np.where(union10['frecuencia_compra'] > 1, 0, 1)

In [ ]:
union10[['frecuencia_compra','frecuencia_binaria']]

In [ ]:
union10.drop(columns=['customer_unique_id', 'frecuencia_compra'], inplace=True)
union10.info()

In [ ]:
conteo_unos = union10['frecuencia_binaria'].sum()

proporcion_No_Vuelve = conteo_unos/95420

print(f'Proporción de clientes "NO VUELVE": {proporcion_No_Vuelve}')

In [ ]:
conteos = union10['frecuencia_binaria'].value_counts()

# Acceder al conteo de 0
conteo_ceros = conteos.get(0, 0)

proporcion_Vuelve=conteo_ceros/95420

print(f'Proporción de clientes "VUELVE": {proporcion_Vuelve}')

Before training the Machine Learning models, the dataset was rebalanced due to the strong class imbalance between returning and non-returning customers.

# Class Balancing

The target variable presented class imbalance. Undersampling techniques were applied to improve model performance during training.

In [ ]:
union10_majority = union10[union10['frecuencia_binaria'] == 1]
union10_minority = union10[union10['frecuencia_binaria'] == 0]



The majority class was undersampled until it represented 80% of the observations, while the minority class remained unchanged.


In [ ]:

desired_majority_size = int(len(union10_minority) * 0.80 / 0.20)
df_majority_downsampled = union10_majority.sample(desired_majority_size, random_state=42)
df_balanced = pd.concat([df_majority_downsampled, union10_minority])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df_balanced.info()

In [ ]:
conteo_unos = df_balanced['frecuencia_binaria'].sum()

proporcion_No_Vuelve = conteo_unos/14565

print(f'Proporción de clientes "NO VUELVE": {proporcion_No_Vuelve}')

In [ ]:
conteos = df_balanced['frecuencia_binaria'].value_counts()

# Acceder al conteo de 0
conteo_ceros = conteos.get(0, 0)

proporcion_Vuelve=conteo_ceros/14565

print(f'Proporción de clientes "VUELVE": {proporcion_Vuelve}')

# Feature Scaling (Min-Max Scaler)

Feature scaling was performed using Min-Max Scaler to normalize numerical variables before model training.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

columnas_a_escalar = ["frecuencia_binaria","monto_pagado", "venta_promedio_estado_cliente", "precio_promedio", "volumen_promedio", "demora_promedio_días", "valor_total_flete"]
scaler = MinMaxScaler()
df_balanced[columnas_a_escalar] = scaler.fit_transform(df_balanced[columnas_a_escalar])
df_balanced.head()

In [ ]:
df_balanced.info()

# Exploratory Visual Analysis

Exploratory visualizations were used to understand feature distributions and identify relationships before model training.

PAIR-PLOT

Feature Correlation Matrix



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

Dummy variables and the average sales by state feature were excluded from the exploratory visual analysis.

In [ ]:

# Graficar todas las combinaciones de features y colorear por 'frecuencia_binaria'
sns.pairplot(df_balanced, vars=["monto_pagado", "precio_promedio",
                         "volumen_promedio", "demora_promedio_días", "valor_total_flete"],
             hue="frecuencia_binaria", markers=["o", "s"])
plt.show()

### Feature Distributions

In [ ]:


# Calcular la matriz de correlación
cor = df_balanced[["monto_pagado", "venta_promedio_estado_cliente", "boleto", "tarjeta_crédito",
            "tarjeta_débito", "voucher", "precio_promedio", "volumen_promedio",
            "demora_promedio_días", "valor_total_flete"]].corr()

# Graficar el heatmap de la matriz de correlación
plt.figure(figsize=(10, 8))
sns.heatmap(cor, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Matriz de correlación de las variables")
plt.show()

# Machine Learning Models

Several supervised Machine Learning algorithms were trained and compared to identify the most effective model for predicting customer recurrence.

**Hypothesis**

Most Olist customers make only one purchase during the analyzed period. Targeted marketing strategies could increase customer retention and repeat purchases.


## Decision Trees

### Single Decision Tree

We begin with a **Decision Tree** model using the **scaled dataset**, although feature scaling is not a requirement for tree-based algorithms. Decision Trees are well suited for datasets containing a mix of **continuous and binary variables**. They are also highly interpretable, making them an excellent choice for communicating results to non-technical stakeholders. The tree visualization further enhances the interpretability of the model.

Decision Trees recursively partition the feature space by selecting splits that maximize the reduction in **impurity** (or equivalently, information entropy), producing increasingly homogeneous observations within each terminal node.

One limitation of Decision Trees is their sensitivity to **class imbalance**. When one class dominates the dataset, the initial impurity is already relatively low, leaving less room for improvement during the splitting process. As a result, the tree may produce nodes that are almost pure and predominantly contain observations from the majority class.

To reduce the risk of **overfitting**, we applied **pre-pruning** by limiting the maximum tree depth to **4**. Without this constraint, the tree could continue splitting until it nearly perfectly fits the training data, resulting in poor generalization to unseen observations.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
datosX = df_balanced[["monto_pagado", "venta_promedio_estado_cliente", "boleto","tarjeta_crédito","tarjeta_débito","voucher","precio_promedio","volumen_promedio","demora_promedio_días","valor_total_flete"]]
X = np.array(datosX)
y = df_balanced['frecuencia_binaria'].values

### Train-Test Split

When splitting the dataset into **training (60%)** and **test (40%)** sets, the original **80:20 class distribution** between **"Does Not Return"** and **"Returns"** was preserved through **stratified sampling**.

Maintaining the class proportions in both subsets ensures that the training and test sets remain representative of the original dataset, reducing potential bias during model training and evaluation.

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.4,random_state=102, stratify=y)

In [ ]:
tree = DecisionTreeClassifier(max_depth=4, random_state=0)

In [ ]:
tree.fit(X_train, Y_train)

In [ ]:
print("Accuracy on training set: {:3f}".format(tree.score(X_train,Y_train)))
print("Accuracy on test set: {:3f}".format(tree.score(X_test,Y_test)))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

plt.figure(figsize=(45, 15))
plot_tree( tree, feature_names=["monto_pagado", "venta_promedio_estado_cliente", "boleto", "tarjeta_crédito", "tarjeta_débito", "voucher", "precio_promedio", "volumen_promedio", "demora_promedio_días", "valor_total_flete"],
    class_names=["Vuelve", "NoVuelve"], filled=True, rounded=True, impurity=False, fontsize=14)
plt.show()


In [ ]:
print("Importancia de cada variable:\n {}".format(tree.feature_importances_))

In [ ]:
print(datosX.columns)

In [ ]:
def plot_features_importance(model, datosX):
    n_features = datosX.shape[1]
    plt.barh(range(n_features), model.feature_importances_, align='center')
    plt.yticks(np.arange(n_features), datosX.columns)
    plt.xlabel("Feature Importance")
    plt.ylabel("Feature")
    plt.title("Importancia de las características")
    plt.show()


plot_features_importance(tree, datosX)

Feature importance analysis showed that shipping cost, average sales by state and product volume were among the strongest predictors of customer recurrence.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.metrics import ConfusionMatrixDisplay


In [ ]:
pred_tree=tree.predict(X_test)


In [ ]:
print("\nMatriz de confusión para el Árbol de Decisión: ")
print(confusion_matrix(Y_test, pred_tree))

In [ ]:
ConfusionMatrixDisplay.from_predictions(Y_test, pred_tree,normalize='true')

### Confusion Matrix Analysis





In [ ]:
print(classification_report(Y_test, pred_tree, target_names=["Vuelve","No Vuelve"]))

### Random Forest

Random Forest achieved the best predictive performance and provided feature importance scores that helped explain the business drivers of customer retention.





In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
forest = RandomForestClassifier(n_estimators=5, random_state=2)
forest.fit(X_train,Y_train)

In [ ]:
print("Accuracy on training set: {:3f}".format(forest.score(X_train,Y_train)))
print("Accuracy on test set: {:3f}".format(forest.score(X_test,Y_test)))

In [ ]:
print("Importancia de cada variable:\n {}".format(forest.feature_importances_))

In [ ]:
def plot_features_importance(model, datosX):
    n_features = datosX.shape[1]
    plt.barh(range(n_features), model.feature_importances_, align='center')
    plt.yticks(np.arange(n_features), datosX.columns)
    plt.xlabel("Feature Importance")
    plt.ylabel("Feature")
    plt.title("Importancia de las características")
    plt.show()


plot_features_importance(forest, datosX)

In [ ]:
pred_forest=forest.predict(X_test)

In [ ]:
print("\nMatriz de confusión para el método RANDOM FOREST: ")
print(confusion_matrix(Y_test, pred_forest))

In [ ]:
ConfusionMatrixDisplay.from_predictions(Y_test, pred_forest,normalize='true')

In [ ]:
print(classification_report(Y_test, pred_forest, target_names=["Vuelve","No Vuelve"]))

Random Forest improved model performance compared with a single Decision Tree. Feature importance analysis confirmed that shipping cost remained the strongest predictor of customer recurrence.

## K-Nearest Neighbors (KNN)

We begin by training the simplest KNN model, using **one nearest neighbor (`k = 1`)**. In this setting, each new observation is assigned the class label of its **closest training instance** according to the selected distance metric.

This baseline model provides a useful reference point for evaluating the impact of different hyperparameter configurations.

After assessing the baseline performance, we perform **hyperparameter tuning** to identify the optimal KNN configuration. In particular, we explore different values of **`k`** and other relevant parameters to improve the model's predictive performance and generalization ability.



### K = 1

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
vecino = KNeighborsClassifier(n_neighbors=1)
vecino.fit(X_train, Y_train)

In [ ]:
pred_vecino=vecino.predict(X_test)

In [ ]:
print("Accuracy on training set: {:3f}".format(vecino.score(X_train,Y_train)))
print("Accuracy on test set: {:3f}".format(vecino.score(X_test,Y_test)))
print("\nMatriz de confusión para el método KNN con K=1: ")
print(confusion_matrix(Y_test, pred_vecino))

In [ ]:
ConfusionMatrixDisplay.from_predictions(Y_test, pred_vecino,normalize='true')

## Hyperparameter Tuning (Optimal *k*)

To identify the optimal number of neighbors, we performed **hyperparameter tuning** using **Cross-Validation**.

The search evaluated values of **`k` ranging from 1 to 15**, training and validating multiple KNN models to determine the configuration with the best predictive performance.

To reduce computational cost, we used **Halving Cross-Validation**, an efficient search strategy that progressively eliminates less promising hyperparameter configurations while allocating more computational resources to the most promising candidates.

Additionally, **Randomized Halving Cross-Validation** introduces a stochastic element into the hyperparameter search by evaluating a random subset of candidate configurations, providing an efficient balance between computational efficiency and model performance.

In [ ]:
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV

In [ ]:
parameters = [{'algorithm': ["auto", "brute"],
              'metric': ["cityblock", "cosine", "euclidean", "haversine", "l1", "l2", "manhattan", "nan_euclidean"],
              'n_neighbors': np.arange(1,15),
              'weights': ["uniform", "distance"]},
              {'algorithm': ["kd_tree", "ball_tree"],
              'metric': ["cityblock", "euclidean", "l1", "l2", "manhattan"],
              'n_neighbors': np.arange(1,15),
              'weights': ["uniform", "distance"]}
              ]

In [ ]:
vecinos_cv = HalvingRandomSearchCV(vecino, parameters, scoring='accuracy', random_state=0)
vecinos_cv.fit(X_train, Y_train)

In [ ]:
best_params = vecinos_cv.best_params_
best_score = vecinos_cv.best_score_

print(best_params, best_score)

In [ ]:
pred_vecinos = vecinos_cv.predict(X_test)

In [ ]:
print("Accuracy on training set: {:3f}".format(vecinos_cv.score(X_train,Y_train)))
print("Accuracy on test set: {:3f}".format(vecinos_cv.score(X_test,Y_test)))

In [ ]:
print("\nMatriz de confusión para el método KNN con K=8: ")
print(confusion_matrix(Y_test, pred_vecinos))

In [ ]:

ConfusionMatrixDisplay.from_predictions(Y_test, pred_vecinos, normalize='true')

In [ ]:
print(classification_report(Y_test, pred_vecinos, target_names=["Vuelve","No Vuelve"]))

## KNN Comparison (*k* = 1 vs. *k* = 8)

The K-Nearest Neighbors (KNN) algorithm has two key hyperparameters: the **number of neighbors (`k`)** and the **distance metric** used to identify the nearest observations.

We first trained a **baseline model** with **`k = 1`**. Then, using **cross-validation**, we optimized the model by selecting both the optimal value of **`k`** and the most appropriate distance metric. The hyperparameter search identified **`k = 8`** as the best-performing configuration.

Compared with the baseline model, the optimized KNN showed a **slight decrease in training accuracy** but an **improvement in test accuracy**. This behavior suggests better **generalization performance**, indicating that the optimized model is less prone to overfitting and is more likely to perform well on unseen data.



## KNN vs. Random Forest

The results obtained so far indicate that **ensemble methods** outperform their simpler counterparts. Just as KNN achieved better performance by considering multiple neighbors instead of only one, **Random Forest** improved upon a single Decision Tree by combining the predictions of multiple trees.

Comparing the two optimized models, **Random Forest** achieved **better performance on the test set** and a **higher precision score** than KNN.

Since one of the main objectives of this project is to **minimize false positives**, **precision** is a particularly important evaluation metric. A higher precision indicates that, among the customers predicted to return, a larger proportion actually do return, reducing the number of false positive predictions and making the model more reliable for business decision-making.

## Linear Models




## Perceptron Algorithm

The **Perceptron** algorithm was evaluated as a **baseline linear classifier** for comparison with the other machine learning models.

The Perceptron is designed for **linearly separable classification problems**, where the objective is to learn a **decision boundary (hyperplane)** that correctly separates the classes.

For this implementation, we specified a **maximum number of iterations (`max_iter`)** as the stopping criterion to ensure convergence during training. Together with the **learning rate**, this is one of the most influential hyperparameters controlling the learning process. In this analysis, the learning rate was left at its **default value**.


In [ ]:
import sklearn.linear_model


In [ ]:
perceptron = sklearn.linear_model.Perceptron(max_iter=10, random_state=1)
perceptron.fit(X_train, Y_train)

pred_perceptron = perceptron.predict(X_test)



In [ ]:
print("Accuracy on training set: {:3f}".format(perceptron.score(X_train,Y_train)))
print("Accuracy on test set: {:3f}".format(perceptron.score(X_test,Y_test)))

In [ ]:
print(classification_report(Y_test, pred_perceptron, target_names=["Vuelve","No Vuelve"]))

## Support Vector Machine (SVM)

Next, we train a **Linear Support Vector Classifier (LinearSVC)**, a linear classification algorithm based on the principles of **Support Vector Machines (SVMs)**.

The objective of a linear SVM is to identify the **optimal separating hyperplane** that maximizes the margin between classes, leading to improved generalization on unseen data. Unlike the classical SVM implementation, **LinearSVC** is computationally more efficient for large datasets and offers greater flexibility for solving linearly separable or approximately linearly separable classification problems.

This model serves as an additional baseline for comparison with the previously evaluated classification algorithms.

In [ ]:
svm = sklearn.svm.LinearSVC()


In [ ]:
svm.fit(X_train, Y_train)

In [ ]:
pred_svm = svm.predict(X_test)

In [ ]:
print("Accuracy on training set: {:3f}".format(svm.score(X_train,Y_train)))
print("Accuracy on test set: {:3f}".format(svm.score(X_test,Y_test)))

In [ ]:
print(classification_report(Y_test, pred_svm, target_names=["Vuelve","No Vuelve"]))

### Kernel-Based SVM

Kernel functions transform the feature space, allowing Support Vector Machines to classify observations that are not linearly separable.

In [ ]:
from sklearn.svm import SVC

## Grid Search for Hyperparameter Optimization

The hyperparameters **`C`** and **`gamma`** were tuned together in order to find an appropriate balance between **overfitting** and **underfitting**.

In general, very high values of both parameters may lead to **overfitting**, while very low values may produce an overly simple model that fails to capture relevant patterns in the data.

The **`C` parameter** controls the penalty for misclassification. A high value of `C` forces the model to classify training observations as correctly as possible, allowing fewer errors and potentially adapting too closely to the training data.

The **`gamma` parameter** controls the influence of each individual observation on the decision boundary. A high value of `gamma` means that each point has a smaller radius of influence, producing a more complex and flexible decision boundary that may overfit the training data.

To reduce computational time, Grid Search was performed using three-fold Cross Validation instead of five-fold Cross Validation, producing equivalent results.

In [ ]:
#En formato texto por la demora para procesar
#from sklearn.model_selection import GridSearchCV
#modelo = SVC()

#param_grid = {
#    'C': [0.001, 0.01, 1, 10, 100],
#   'gamma': [0.001, 0.01, 1, 10, 100]
#}
#grid_search = GridSearchCV(estimator=modelo, param_grid=param_grid, cv=3, scoring='accuracy', verbose=1)

#grid_search.fit(X_train, Y_train)

#best_score = grid_search.best_score_
#best_parameters = grid_search.best_params_


#print("Best score:", best_score)
#print("Best parameters:", best_parameters)


#### Sigmoid Kernel

In [ ]:
SVM = SVC(kernel='sigmoid', C = 100 , gamma= 100).fit(X_train, Y_train)

In [ ]:
pred_SVM = SVM.predict(X_test)

In [ ]:
print("Accuracy on training set: {:3f}".format(SVM.score(X_train,Y_train)))
print("Accuracy on test set: {:3f}".format(SVM.score(X_test,Y_test)))

In [ ]:
print(classification_report(Y_test, pred_SVM, target_names=["Vuelve","No Vuelve"]))

### RBF Kernel

The **Radial Basis Function (RBF)**, also known as the **Gaussian kernel**, enables Support Vector Machines to model **nonlinear relationships** by implicitly mapping the data into a higher-dimensional feature space.

This transformation allows the classifier to find a linear separating hyperplane in the transformed space, even when the observations are **not linearly separable in the original feature space**.

The RBF kernel is one of the most widely used kernels because of its flexibility and ability to capture complex decision boundaries while maintaining strong predictive performance.

In [ ]:
SVMrbf = SVC(kernel='rbf', C = 100 , gamma= 100, probability=True).fit(X_train, Y_train)

In [ ]:
pred_SVMrbf = SVMrbf.predict(X_test)

In [ ]:
print("Accuracy on training set: {:3f}".format(SVMrbf.score(X_train,Y_train)))
print("Accuracy on test set: {:3f}".format(SVMrbf.score(X_test,Y_test)))

In [ ]:
print(classification_report(Y_test, pred_SVMrbf, target_names=["Vuelve","No Vuelve"]))

Using the RBF kernel improved both Accuracy and Precision compared with the linear model.

## ROC Curve: Random Forest vs. SVM

**Random Forest** and **Support Vector Machine (SVM) with an RBF kernel** were selected for comparison because they achieved the highest predictive performance among the evaluated models.

The **Receiver Operating Characteristic (ROC) curve** was used to assess and compare the classification performance of both models across all possible classification thresholds.

The ROC curve is constructed by plotting the **True Positive Rate (Recall/Sensitivity)** against the **False Positive Rate** for every possible decision threshold.

The **Area Under the ROC Curve (AUC)** provides a threshold-independent measure of model performance. It can be interpreted as the probability that the classifier assigns a higher score to a randomly selected positive instance than to a randomly selected negative instance.

An effective classifier produces a ROC curve that approaches the **upper-left corner** of the plot, indicating **high sensitivity** while maintaining a **low false positive rate**. Consequently, models with **higher AUC values** are considered to have better overall discriminative ability.

## Model Comparison

The final comparison was conducted between **Random Forest** and **Support Vector Machine (SVM) with an RBF kernel**, as both models achieved the **highest test accuracy** while maintaining **high precision**, making them the strongest candidates for the customer return prediction task.

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, auc, roc_auc_score
from sklearn.datasets import make_classification


In [ ]:

forest_probs = forest.predict_proba(X_test)[:, 1]
SVMrbf_probs = SVMrbf.predict_proba(X_test)[:, 1]

In [ ]:
rf_fpr, rf_tpr, _= roc_curve(Y_test, forest_probs)
svm_fpr, svm_tpr, _ = roc_curve(Y_test, SVMrbf_probs)

In [ ]:
rf_roc_auc = roc_auc_score(Y_test, forest_probs)
svm_roc_auc = roc_auc_score(Y_test, SVMrbf_probs)

In [ ]:
rf_precision, rf_recall, _ = precision_recall_curve(Y_test, forest_probs)
svm_precision, svm_recall, _ = precision_recall_curve(Y_test, SVMrbf_probs)

rf_pr_auc = auc(rf_recall, rf_precision)
svm_pr_auc = auc(svm_recall, svm_precision)

In [ ]:
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(rf_fpr, rf_tpr, label=f'Random Forest (AUC = {rf_roc_auc:.2f})')
plt.plot(svm_fpr, svm_tpr, label=f'SVM (AUC = {svm_roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('TASA DE FALSOS POSITIVOS')
plt.ylabel('TASA DE VERDADEROS POSITIVOS')
plt.title('CURVA ROC')
plt.legend()


## Precision–Recall Curve: Random Forest vs. SVM

The **Precision–Recall (PR) curve** complements the ROC analysis, particularly in the presence of **class imbalance**, where it provides a more informative evaluation of classification performance.

The PR curve is obtained by plotting **Precision** against **Recall** across all possible classification thresholds. An ideal classifier produces a curve that approaches the **upper-right corner**, indicating both high precision and high recall.

Given Olist's business objective, **precision** is the most relevant performance metric. The goal is to **minimize false positives**, thereby avoiding unnecessary marketing expenditures on customers predicted as **"Does Not Return"** who would actually have returned and made another purchase without intervention.

A model with higher precision allows retention campaigns to be targeted more efficiently, reducing marketing costs while maximizing the return on customer retention efforts.

In [ ]:
plt.subplot(1, 2, 2)
plt.plot(rf_recall, rf_precision, label=f'Random Forest (AUC = {rf_pr_auc:.2f})')
plt.plot(svm_recall, svm_precision, label=f'SVM (AUC = {svm_pr_auc:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()

plt.tight_layout()
plt.show()

# Conclusions





Several supervised Machine Learning models were evaluated using performance metrics, ROC curves and Precision–Recall curves.

Among the evaluated models, **Random Forest** achieved the best overall performance for predicting customer recurrence. In addition to its predictive accuracy, the model provides feature importance scores, revealing that **shipping cost** was the most influential variable associated with customer non-recurrence.

## Business Recommendations

The predictive model can be integrated into the purchasing process to estimate whether a customer is likely to return after completing an order.

Customers classified as **high risk of non-recurrence** could receive targeted retention actions such as:

* Personalized discounts on future purchases or shipping costs.
* Product recommendations based on previous purchases.
* Automated follow-up email campaigns after a predefined period.
* Loyalty programs with rewards for repeat customers.
* A/B testing strategies combining product prices and shipping costs to optimize customer retention.

These recommendations require relatively low implementation costs compared with structural logistics changes, while providing actionable insights that can improve customer retention and increase customer lifetime value.

The project demonstrates how Business Analytics and Machine Learning can support data-driven decision making by transforming transactional data into practical business strategies.
